# CSIP Module 12: Uncertainty Quantification & Extrapolation Guardrails
**Computational Society Intelligence Platform (CSIP) — Behavioral Decision Intelligence (BDI)**

### Metodologi Data Science Data Science Competency Standards
- **[UK-10 Mengevaluasi Hasil Pemodelan]**: Kuantifikasi ketidakpastian proyeksi simulasi via Multi-Trajectory Residual Bootstrapping.
- **Claim Governance**: Deteksi Out-of-Distribution (OOD) dan penurunan Evidence Grade otomatis (Grade B ke Grade C/D).

---


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

results_path = REPO_ROOT / 'artifacts' / 'reports' / 'scenario_simulation_results.json'
with open(results_path, 'r') as f:
    sim_data = json.load(f)

print('Loaded simulation uncertainty profiles.')

Loaded simulation uncertainty profiles.


## 1. Prediction Interval Ordering & Dispersion Analysis
We verify that the generated Prediction Intervals satisfy the monotonic ordering:
$$P_{02.5} \le P_{05.0} \le P_{50.0} (\text{Median}) \le P_{95.0} \le P_{97.5}$$

In [2]:
for s_id in ['S0_baseline', 'S2_carrier_sla', 'S4_macro_shock']:
    s = sim_data[s_id]
    intv = s['intervals']
    vol_p025 = np.array(intv['p025'])[1:, 0]
    vol_p50 = np.array(intv['median'])[1:, 0]
    vol_p975 = np.array(intv['p975'])[1:, 0]
    
    is_monotonic = bool(np.all(vol_p025 <= vol_p50) and np.all(vol_p50 <= vol_p975))
    print(f"Scenario {s_id:20s}: Monotonic PI Ordering Verified = {is_monotonic}")
    print(f"  - Week 12 Volume 95% PI: [{vol_p025[-1]:.0f}, {vol_p975[-1]:.0f}] (Median: {vol_p50[-1]:.0f})")

Scenario S0_baseline         : Monotonic PI Ordering Verified = True
  - Week 12 Volume 95% PI: [299, 2354] (Median: 1039)
Scenario S2_carrier_sla      : Monotonic PI Ordering Verified = True
  - Week 12 Volume 95% PI: [168, 2170] (Median: 985)
Scenario S4_macro_shock      : Monotonic PI Ordering Verified = True
  - Week 12 Volume 95% PI: [152, 2095] (Median: 1010)


## 2. Extrapolation Guardrail Audit
We audit which scenarios stay within historical convex bounds and which trigger OOD warnings.

In [3]:
guardrail_rows = []
for s_id, s in sim_data.items():
    g = s['guardrail']
    guardrail_rows.append({
        'Scenario': s_id,
        'Out of Distribution': g['is_out_of_distribution'],
        'Max Z-Score': round(g['max_z_score'], 2),
        'Evidence Grade': g['evidence_grade'],
        'Violations Count': len(g['violations']),
    })

display(pd.DataFrame(guardrail_rows))

,Scenario,Out of Distribution,Max Z-Score,Evidence Grade,Violations Count
0,S0_baseline,False,1.86,Grade B (Cukup / Bersyarat),0
1,S1_freight_subsidy,False,1.86,Grade B (Cukup / Bersyarat),0
2,S2_carrier_sla,False,2.75,Grade B (Cukup / Bersyarat),0
3,S3_targeted_friction,False,1.86,Grade B (Cukup / Bersyarat),0
4,S4_macro_shock,True,4.14,Grade C (Spekulatif / Guardrail Triggered),1
5,S5_combined_blitz,False,2.51,Grade B (Cukup / Bersyarat),0
